## Atividade 4: Modelagem com Multilayer Perceptron (MLP)

### 1) Dados:

Utilize o conjunto de dados normalizados (com MinMaxScaler), conforme utilizou na tarefa anterior.

### 2) Definição do Modelo MLP:

O modelo MLPRegressor é uma rede neural feedforward de múltiplas camadas, com os seguintes parâmetros principais:

- n_hidden_layers: define o número de camadas ocultas.
- hidden_layers_size: define o número de neurônios em cada camada oculta.
- activation: função de ativação (ReLU, tanh, logistic, etc.).
- learning_rate_init: taxa de aprendizado inicial.
- solver: algoritmo de otimização (Podem testar com 'adam' e 'nadam')
- alpha: regularização L2.

Obs.: Regularização é uma técnica usada para evitar overfitting em modelos de aprendizado de máquina, adicionando uma penalização nos pesos na função de custo. 

### 3) Otimização dos hiperparâmetros:

Use o Optuna (ou Gridsearch) para otimizar os parâmetros do item 2. Use, por exemplo:

- n_hidden_layers: entre 1 e 4.
- hidden_layers_size: entre o número de neurônios de entrada (features) e 32.
- activation: ReLU e tanh.
- learning_rate_init: entre 1e-5 e 1e-2.
- solver:  adam e nadam
- alpha: regularização L2.

A regularização L2 é um hiperparâmetro que controla o “peso” da penalização. Se for muito pequeno, o modelo pode sobreajustar (overfit); se for muito grande, o modelo pode subajustar (underfit). Peça para o Optuna buscar o valor ideal dentro do intervalo 1e-6, 1e-2, por exemplo.

### 4) Treine o modelo final com os melhores hyperparâmetros e compare os resultados com o modelo Random Forest. 

### 1: Import das bibliotecas

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
import optuna
from optuna.samplers import TPESampler
from sklearn.exceptions import ConvergenceWarning
import warnings
import os
import glob
import pickle
import joblib

warnings.filterwarnings("ignore", category=ConvergenceWarning)


run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
print(f"Run ID: {run_id}")

### 2: Dados

In [ ]:
x_train = pd.read_csv('data/X_train_scaled.csv', index_col=0)
y_train = pd.read_csv('data/y_train_scaled.csv', index_col=0)

x_val = pd.read_csv('data/X_val_scaled.csv', index_col=0)
y_val = pd.read_csv('data/y_val_scaled.csv', index_col=0)

x_test = pd.read_csv('data/X_test_scaled.csv', index_col=0)
y_test = pd.read_csv('data/y_test_scaled.csv', index_col=0)

RANDOM_SEED = 27
np.random.seed(RANDOM_SEED)

def latest_file(pattern):
    files = glob.glob(pattern)
    return max(files, key=os.path.getmtime) if files else None

scaler_y_path = latest_file("data/scaler_y_*.pkl")
if scaler_y_path is None:
    raise FileNotFoundError("Scaler do alvo não encontrado (data/scaler_y_*.pkl). Rode o notebook 02.")
with open(scaler_y_path, "rb") as f:
    scaler_y = pickle.load(f)
print(f"Scaler y carregado: {os.path.basename(scaler_y_path)}")

### 3: Definição do Modelo MLP

In [ ]:
mlp = MLPRegressor(
    hidden_layer_sizes=(100, 50),
    activation='relu',
    solver='adam',
    learning_rate='adaptive',
    max_iter=500,
    random_state=RANDOM_SEED
)

mlp.fit(x_train, y_train.values.ravel())

val_predictions = mlp.predict(x_val)

rmse = root_mean_squared_error(y_val, val_predictions)
mae = mean_absolute_error(y_val, val_predictions)
r2 = r2_score(y_val, val_predictions)

print(f'Validação -> RMSE: {rmse:.6f} | MAE: {mae:.6f} | R²: {r2:.6f}')

### 4: Otimização dos hiperparâmetros

In [ ]:
n_features = x_train.shape[1]

def objective(trial):
    # Espaço de busca
    n_hidden_layers = trial.suggest_int("n_hidden_layers", 1, 4)
    hidden_units = trial.suggest_int(
        "hidden_units",
        low=min(n_features, 32),
        high=max(n_features, 32)
    )
    activation = trial.suggest_categorical("activation", ["relu", "tanh"])
    learning_rate_init = trial.suggest_float("learning_rate_init", 1e-5, 1e-2, log=True)
    alpha = trial.suggest_float("alpha", 1e-6, 1e-2, log=True)
    solver = "adam"

    hidden_layer_sizes = tuple([hidden_units] * n_hidden_layers)

    model = MLPRegressor(
        hidden_layer_sizes=hidden_layer_sizes,
        activation=activation,
        solver=solver,
        learning_rate="adaptive",
        learning_rate_init=learning_rate_init,
        alpha=alpha,
        max_iter=1000,
        early_stopping=True,
        n_iter_no_change=20,
        random_state=RANDOM_SEED
    )

    model.fit(x_train, y_train.values.ravel())
    preds_val = model.predict(x_val)

    rmse = root_mean_squared_error(y_val, preds_val)
    return rmse  # Optuna minimiza

study = optuna.create_study(direction="minimize", sampler=TPESampler(seed=RANDOM_SEED))
study.optimize(objective, n_trials=30, show_progress_bar=True, n_jobs=-1)

print("Melhor RMSE (val):", study.best_value)
print("Melhores parâmetros:")
for k, v in study.best_params.items():
    print(f"  - {k}: {v}")
    
trials_df = study.trials_dataframe()
trials_df.to_csv(f"data/mlp_optuna_trials_{run_id}.csv", index=False)

pd.DataFrame([study.best_params]).to_csv(f"data/mlp_best_params_{run_id}.csv", index=False)

In [ ]:
# (Opcional) Treina o modelo final com os melhores hiperparâmetros e avalia em validação e teste

try:
    run_id
except NameError:
    run_id = datetime.now().strftime("%Y%m%d_%H%M%S")

# Função utilitária
def latest_file(pattern):
    files = glob.glob(pattern)
    return max(files, key=os.path.getmtime) if files else None

# Obtém melhores parâmetros:
best_params = None
source = None

if "study" in globals() and hasattr(study, "best_params"):
    best_params = study.best_params
    source = "study.em.memoria"
else:
    bp_path = latest_file("data/mlp_best_params_*.csv")
    if bp_path:
        df_bp = pd.read_csv(bp_path)
        # Assume primeira linha
        best_params = df_bp.iloc[0].to_dict()
        source = f"csv:{os.path.basename(bp_path)}"

# Fallback simples caso não encontre
if best_params is None:
    best_params = {
        "n_hidden_layers": 2,
        "hidden_units": 64,
        "activation": "relu",
        "learning_rate_init": 1e-3,
        "alpha": 1e-4
    }
    source = "fallback.default"

print(f"Fonte dos hiperparâmetros: {source}")
print("Hiperparâmetros usados:")
for k, v in best_params.items():
    print(f"  {k}: {v}")

# Constrói tupla de camadas
n_layers = int(best_params["n_hidden_layers"])
hidden_units = int(best_params["hidden_units"])
hidden_layer_sizes = tuple([hidden_units] * n_layers)

mlp_best = MLPRegressor(
    hidden_layer_sizes=hidden_layer_sizes,
    activation=best_params.get("activation", "relu"),
    solver="adam",
    learning_rate="adaptive",
    learning_rate_init=float(best_params.get("learning_rate_init", 1e-3)),
    alpha=float(best_params.get("alpha", 1e-4)),
    max_iter=1000,
    early_stopping=True,
    n_iter_no_change=20,
    random_state=RANDOM_SEED
)

# Treina no conjunto de treino
mlp_best.fit(x_train, y_train.values.ravel())

# Métricas em validação
val_pred = mlp_best.predict(x_val)
val_rmse = root_mean_squared_error(y_val, val_pred)
val_mae = mean_absolute_error(y_val, val_pred)
val_r2 = r2_score(y_val, val_pred)
print(f"Validação -> RMSE: {val_rmse:.6f} | MAE: {val_mae:.6f} | R²: {val_r2:.6f}")

# Re-treina no conjunto treino+validação para obter o modelo final antes do teste
x_tr_full = pd.concat([x_train, x_val], axis=0)
y_tr_full = pd.concat([y_train, y_val], axis=0)

mlp_best.fit(x_tr_full, y_tr_full.values.ravel())

# Métricas em teste
test_pred = mlp_best.predict(x_test)
test_rmse = root_mean_squared_error(y_test, test_pred)
test_mae = mean_absolute_error(y_test, test_pred)
test_r2 = r2_score(y_test, test_pred)
print(f"Teste     -> RMSE: {test_rmse:.6f} | MAE: {test_mae:.6f} | R²: {test_r2:.6f}")

# Métricas em escala original (mm) via inverse_transform

y_val_mm  = scaler_y.inverse_transform(y_val.values)
y_test_mm = scaler_y.inverse_transform(y_test.values)
val_pred_mm  = scaler_y.inverse_transform(val_pred.reshape(-1, 1))
test_pred_mm = scaler_y.inverse_transform(test_pred.reshape(-1, 1))

val_rmse_mm  = root_mean_squared_error(y_val_mm,  val_pred_mm)
val_mae_mm   = mean_absolute_error(y_val_mm, val_pred_mm)
val_r2_mm    = r2_score(y_val_mm, val_pred_mm)

test_rmse_mm = root_mean_squared_error(y_test_mm, test_pred_mm)
test_mae_mm  = mean_absolute_error(y_test_mm, test_pred_mm)
test_r2_mm   = r2_score(y_test_mm, test_pred_mm)

print(f"Validação (mm) -> RMSE: {val_rmse_mm:.3f} | MAE: {val_mae_mm:.3f} | R²: {val_r2_mm:.4f}")
print(f"Teste     (mm) -> RMSE: {test_rmse_mm:.3f} | MAE: {test_mae_mm:.3f} | R²: {test_r2_mm:.4f}")

# CSV simples com métricas em mm
metrics_mm_df = pd.DataFrame(
    [
        {"split": "val",  "rmse": float(val_rmse_mm),  "mae": float(val_mae_mm),  "r2": float(val_r2_mm)},
        {"split": "test", "rmse": float(test_rmse_mm), "mae": float(test_mae_mm), "r2": float(test_r2_mm)},
    ]
)
metrics_mm_df.to_csv(f"data/mlp_final_metrics_mm_{run_id}.csv", index=False)

metrics_df = pd.DataFrame(
    [
        {"split": "val", "rmse": float(val_rmse), "mae": float(val_mae), "r2": float(val_r2)},
        {"split": "test", "rmse": float(test_rmse), "mae": float(test_mae), "r2": float(test_r2)},
    ]
)
metrics_df.to_csv(f"data/mlp_final_metrics_{run_id}.csv", index=False)

### 5: Comparação entre MLP (Optuna) e Random Forest

Nesta seção, comparamos o desempenho dos modelos no conjunto de teste.
- Carrega as métricas finais do MLP a partir do CSV salvo.
- Carrega o modelo final de Random Forest salvo em disco e calcula as métricas no teste.
- Exibe uma tabela lado a lado e gráficos de barras para RMSE/MAE (menor é melhor) e R² (maior é melhor).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

# Se seu RF foi treinado com alvo não escalado, mude para False
rf_target_is_scaled = True

def latest_or_none(patterns):
    paths = []
    for p in patterns:
        paths.extend(glob.glob(p))
    return max(paths, key=os.path.getmtime) if paths else None

# Carrega scaler_y (para trazer tudo para mm)
scaler_y_path = latest_or_none(["data/scaler_y_*.pkl"])
if scaler_y_path is None:
    raise FileNotFoundError("Scaler do alvo (data/scaler_y_*.pkl) não encontrado.")
with open(scaler_y_path, "rb") as f:
    scaler_y = pickle.load(f)

# MLP: usa mlp_best em memória, prevê e converte para mm
mlp_pred_scaled = mlp_best.predict(x_test)
y_test_mm = scaler_y.inverse_transform(y_test.values)
mlp_pred_mm = scaler_y.inverse_transform(mlp_pred_scaled.reshape(-1, 1))
mlp_rmse_mm = root_mean_squared_error(y_test_mm, mlp_pred_mm)
mlp_mae_mm = mean_absolute_error(y_test_mm, mlp_pred_mm)
mlp_r2_mm = r2_score(y_test_mm, mlp_pred_mm)

# Random Forest: carrega o modelo salvo e converte para mm se necessário
rf_model_path = latest_or_none([f"modelo_rf_{run_id}.pkl", "modelo_rf_*.pkl", "modelo_rf.pkl"])
if rf_model_path is None:
    raise FileNotFoundError("Modelo Random Forest não encontrado. Rode o notebook 03 e salve modelo_rf*.pkl.")
rf = joblib.load(rf_model_path)
rf_pred = rf.predict(x_test)
if rf_target_is_scaled:
    rf_pred_mm = scaler_y.inverse_transform(rf_pred.reshape(-1, 1))
else:
    rf_pred_mm = rf_pred.reshape(-1, 1)

rf_rmse_mm = root_mean_squared_error(y_test_mm, rf_pred_mm)
rf_mae_mm = mean_absolute_error(y_test_mm, rf_pred_mm)
rf_r2_mm = r2_score(y_test_mm, rf_pred_mm)

# Tabela de comparação (mm)
comp_df_mm = pd.DataFrame(
    [
        {"model": "MLP (Optuna)", "rmse_mm": mlp_rmse_mm, "mae_mm": mlp_mae_mm, "r2": mlp_r2_mm},
        {"model": "RandomForest", "rmse_mm": rf_rmse_mm, "mae_mm": rf_mae_mm, "r2": rf_r2_mm},
    ]
).set_index("model").round(6)

print("Comparação no Teste (escala original - mm):")
display(comp_df_mm)

# Salva comparação
comp_df_mm.to_csv(f"data/compare_mlp_rf_mm_{run_id}.csv")

# Gráficos (mm)
ax = comp_df_mm[["rmse_mm", "mae_mm"]].plot(kind="bar", figsize=(8, 4), rot=0,
                                            title="Erro no Teste em mm (menor é melhor)")
plt.ylabel("mm")
plt.tight_layout()
plt.show()

ax = comp_df_mm[["r2"]].plot(kind="bar", figsize=(6, 4), rot=0, legend=False,
                             title="R² no Teste (maior é melhor)", color=["tab:green"])
plt.ylim(0, 1.0)
plt.tight_layout()
plt.show()

# Diferença percentual em RMSE (mm)
delta_rmse_pct = (rf_rmse_mm - mlp_rmse_mm) / rf_rmse_mm * 100
print(f"Variação RMSE (MLP vs RF) em mm: {delta_rmse_pct:+.2f}%")
